In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# 确保多进程并行共享GPU
physical_devices = tf.config.experimental.list_physical_devices('GPU')
if physical_devices:
    try:
        for device in physical_devices:
            tf.config.experimental.set_memory_growth(device, True) 
    except RuntimeError as e:
        print(e)


In [ ]:
import os
import sys
import numpy as np
# 假设 dataset0 文件夹在当前工作目录下
module_path = os.path.abspath(os.path.join('./Agrobot Dataset/dataset0'))
if module_path not in sys.path:
    sys.path.append(module_path)
print(os.getcwd()) 

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import atan2, pi, sqrt, cos, sin, floor
import tensorflow as tf
from tensorflow.python.keras.backend import set_session
config = tf.compat.v1.ConfigProto() 
config.gpu_options.allow_growth = True  
config.log_device_placement = True  
sess2 = tf.compat.v1.Session(config=config)
set_session(sess2)  
from tensorflow.keras.layers import Dense, MaxPooling1D, Flatten
from tensorflow.keras import Input, Model
from tensorflow.keras.callbacks import ModelCheckpoint
import tensorflow.keras.backend as K
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_squared_error
from scipy.stats import uniform
import pickle
import csv
import random
import itertools
import math
import time
import sys
import os
from data_utils_0 import *
from traj_utils import *
from neural_ekf import *
from tcn import TCN

## Import Dataset

In [ ]:
# 创建函数以计算 FLOPS
def get_flops(model):
    concrete = tf.function(lambda inputs: model(inputs))
    concrete_func = concrete.get_concrete_function(tf.TensorSpec([1] + model.input_shape[1:], model.inputs[0].dtype))

    frozen_func, graph_def = convert_variables_to_constants_v2_as_graph(concrete_func)

    with tf.Graph().as_default() as graph:
        tf.import_graph_def(graph_def, name='')
        run_meta = tf.compat.v1.RunMetadata()
        opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()

        flops = tf.compat.v1.profiler.profile(graph=graph, run_meta=run_meta, cmd='op', options=opts)
        return flops.total_float_ops

In [ ]:
window_size = 100
stride = 20

f = './Agrobot Dataset/dataset0/'
X_train,Y_Pos_train, Physics_Vec_train, x_vel_train, y_vel_train, x0_list_train, y0_list_train, size_of_each_train = import_agrobot_dataset_p1(dataset_folder=f, type_flag=1, window_size=window_size, stride=stride)
P = np.repeat(Physics_Vec_train,window_size).reshape((Physics_Vec_train.shape[0],window_size,1)) #导入数据集1中的训练集
X_train = np.concatenate((X_train,P),axis=2)

X_test,Y_Pos_test, Physics_Vec_test, x_vel_test, y_vel_test, x0_list_test, y0_list_test, size_of_each_test= import_agrobot_dataset_p1(type_flag = 2, dataset_folder=f,window_size=window_size, stride=stride)
P_test = np.repeat(Physics_Vec_test,window_size).reshape((Physics_Vec_test.shape[0],window_size,1)) #导入数据集1中的测试集
X_test = np.concatenate((X_test,P_test),axis=2)

## Load the Pre-trained Model

In [ ]:
from torchviz import make_dot
model = load_model('./Neural Kalman IMU GNSS Fusion/Agrobot_First_TCN.keras',custom_objects={'TCN':TCN})

## Kalman Model

Seen Trajectory

In [ ]:
ATE = []
RTE = []
for i in range(len(size_of_each_train)): #计算每个训练文件的ATE与RTE，i为每个文件的文件序号

    fused_pos_x, fused_pos_y, GPS_x, GPS_y = neural_ekf_gnss_imu(X_train, x_vel_train,y_vel_train, #输出预测的X、Y位置数据与GPS位置数据
                size_of_each_train,
                x0_list_train, y0_list_train,i,window_size,stride,60*5,
                 model)  # i 作为file_index输入
    act_x,act_y =  GT_pos_generator(x_vel_train, y_vel_train, size_of_each_train, 
                   x0_list_train, y0_list_train, window_size, stride,i) #导入真值数据集
    
    
        
    at, rt, at_all, rt_all = Cal_TE(act_x, act_y, fused_pos_x, fused_pos_y,
                                    sampling_rate=100,window_size=window_size,stride=stride) #计算融合后结果的ATE与RTE

    ATE.append(at)
    RTE.append(rt)
    print('ATE, RTE:',ATE[i],RTE[i])
    
print('Median ATE and RTE', np.median(ATE),np.median(RTE))
print('Mean ATE and RTE', np.mean(ATE),np.mean(RTE))
print('STD ATE and RTE', np.std(ATE),np.std(RTE))

Unseen Trajectory

In [ ]:
ATE = []
RTE = []
for i in range(len(size_of_each_test)): #计算每个测试文件的ATE与RTE，i为指定文件序号

    fused_pos_x, fused_pos_y, GPS_x, GPS_y =  neural_ekf_gnss_imu(X_test, x_vel_test,y_vel_test, #获得N_EKF的输出：X轴位置数据，Y轴位置数据，GPSX轴位置数据,GPSY轴位置数据 
                size_of_each_test,
                x0_list_test, y0_list_test,i,window_size,stride,5,
                 model)
    act_x,act_y =  GT_pos_generator(x_vel_test, y_vel_test, size_of_each_test, 
                   x0_list_test, y0_list_test, window_size, stride,i) #获得真值位置数据
    
    
        
    at, rt, at_all, rt_all = Cal_TE(act_x, act_y, fused_pos_x, fused_pos_y,
                                    sampling_rate=100,window_size=window_size,stride=stride) #ATE与RTE计算

    ATE.append(at)
    RTE.append(rt)
    print('ATE, RTE:',ATE[i],RTE[i])
    #ATE与RTE中包含了所有点值
print('Median ATE and RTE', np.median(ATE),np.median(RTE)) #取ATE与RTE的中位数
print('Mean ATE and RTE', np.mean(ATE),np.mean(RTE)) #取ATE与RTE的平均数
print('STD ATE and RTE', np.std(ATE),np.std(RTE)) #计算ATE与RTE的标准差


Plot

In [ ]:
#画真实轨迹图与预测轨迹图
fused_pos_x, fused_pos_y, GPS_x, GPS_y =  neural_ekf_gnss_imu(X_train, x_vel_train,y_vel_train, 
            size_of_each_train,
            x0_list_train, y0_list_train,0,window_size,stride,5,
             model) #获取指定数据集的N_EKF的预测结果
act_x,act_y =  GT_pos_generator(x_vel_train, y_vel_train, size_of_each_train, 
               x0_list_train, y0_list_train, window_size, stride,0) #获取指定数据集的真实位置数据

In [ ]:
a = 0
b = 3000
plt.plot(act_x[a:b],act_y[a:b],label='Ground Truth',linestyle='-') #画真实位置轨迹
plt.plot(fused_pos_x[a:b],fused_pos_y[a:b],label='Neurl-KF',linestyle='-') #画预测位置轨迹
plt.scatter(GPS_x[math.ceil(a/5):math.ceil(b/5)],GPS_y[math.ceil(a/5):math.ceil(b/5)],
            marker='.',label='GPS only') #画出GPS测试轨迹以作对比
plt.xlim([-2,1.5])
plt.ylim([-2,2])
plt.grid('minor')
plt.xlabel('East (m)')
plt.ylabel('North (m)')
plt.title('Phase 1, with GPS')
plt.legend(loc='best')